In [2]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text

In [3]:
MYSQL_USER = 'root'
MYSQL_PASSWORD = 'Pedro2150!'
MYSQL_HOST = 'localhost'
MYSQL_PORT = 3306
MYSQL_DATABASE = 'olist_analytics'

engine = create_engine(
    f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}?charset=utf8mb4'
)

In [4]:
PROCESSED_PATH = Path('../data/processed')

In [5]:
tables_to_import = [
    ('stg_customers', 'customers_clean.csv'),
    ('stg_sellers', 'sellers_clean.csv'),
    ('stg_geolocation', 'geolocation_clean.csv'),
    ('stg_products', 'products_clean.csv'),
    ('stg_orders', 'orders_clean.csv'),
    ('stg_order_items', 'order_items_clean.csv'),
    ('stg_payments', 'payments_clean.csv'),
    ('stg_reviews', 'reviews_clean.csv')
]

In [6]:
tables_to_clear = [
    'stg_reviews',
    'stg_payments',
    'stg_order_items',
    'stg_orders',
    'stg_products',
    'stg_geolocation',
    'stg_sellers',
    'stg_customers'
]

with engine.begin() as connection:
    connection.execute(text('SET FOREIGN_KEY_CHECKS = 0;'))
    
    for table_name in tables_to_clear:
        connection.execute(text(f'DELETE FROM {table_name};'))
    
    connection.execute(text('SET FOREIGN_KEY_CHECKS = 1;'))

print('Tabelas limpas com sucesso.')

Tabelas limpas com sucesso.


In [7]:
for table_name, file_name in tables_to_import:
    file_path = PROCESSED_PATH / file_name
    
    print(f'Importando {file_name} para {table_name}...')
    
    df = pd.read_csv(file_path)
    
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists='append',
        index=False,
        chunksize=5000,
        method='multi'
    )
    
    print(f'{table_name} importada com sucesso. Linhas: {len(df)}')

print('Importação finalizada.')

Importando customers_clean.csv para stg_customers...
stg_customers importada com sucesso. Linhas: 99441
Importando sellers_clean.csv para stg_sellers...
stg_sellers importada com sucesso. Linhas: 3095
Importando geolocation_clean.csv para stg_geolocation...
stg_geolocation importada com sucesso. Linhas: 19015
Importando products_clean.csv para stg_products...
stg_products importada com sucesso. Linhas: 32951
Importando orders_clean.csv para stg_orders...
stg_orders importada com sucesso. Linhas: 99441
Importando order_items_clean.csv para stg_order_items...
stg_order_items importada com sucesso. Linhas: 112650
Importando payments_clean.csv para stg_payments...
stg_payments importada com sucesso. Linhas: 103886
Importando reviews_clean.csv para stg_reviews...
stg_reviews importada com sucesso. Linhas: 99224
Importação finalizada.


In [8]:
validation_query = """
SELECT 'stg_customers' AS table_name, COUNT(*) AS rows_count FROM stg_customers
UNION ALL
SELECT 'stg_sellers', COUNT(*) FROM stg_sellers
UNION ALL
SELECT 'stg_geolocation', COUNT(*) FROM stg_geolocation
UNION ALL
SELECT 'stg_products', COUNT(*) FROM stg_products
UNION ALL
SELECT 'stg_orders', COUNT(*) FROM stg_orders
UNION ALL
SELECT 'stg_order_items', COUNT(*) FROM stg_order_items
UNION ALL
SELECT 'stg_payments', COUNT(*) FROM stg_payments
UNION ALL
SELECT 'stg_reviews', COUNT(*) FROM stg_reviews;
"""

pd.read_sql(validation_query, engine)

,table_name,rows_count
0,stg_customers,99441
1,stg_sellers,3095
2,stg_geolocation,19015
3,stg_products,32951
4,stg_orders,99441
5,stg_order_items,112650
6,stg_payments,103886
7,stg_reviews,99224
